# SmartPantry AI: Mochi AI Assistant

## Phase 4 - AI Assistant Development and Testing

**Project:** SmartPantry AI  
**Component:** Mochi AI Assistant  
**Python Module:** `modules/ai_assistant.py`  
**Notebook:** `notebooks/ai_assistant.ipynb`

### Overview

Mochi is the conversational AI assistant for SmartPantry AI. Its purpose is to help users interact with food and pantry information through natural language.

This notebook documents the assistant's implementation, functionality, testing, and planned integration with other SmartPantry components.

### Phase 4 Objectives

- Examine the existing AI assistant implementation.
- Test assistant responses using controlled inputs.
- Evaluate how Mochi handles errors and unavailable information.
- Explore integration with pantry inventory, recipe recommendations, storage guidance, and food recall information.
- Document test results, limitations, and future improvements.

### Testing Approach

Initial tests will use mocked or controlled responses to avoid unnecessary paid API requests. Live API testing will be performed only when necessary and clearly identified.

Food safety information must be handled carefully. Mochi should not invent recall results, expiration dates, or storage recommendations.

### Google Colab Setup

When using Google Colab, clone the Phase 4 branch before running
the Python import cells:

`!git clone --branch docs/phase4-setup https://github.com/SmartPantry-AI/smartpantry-ai.git`

In VS Code, this step is unnecessary because the repository
is already available locally.

## 1. Existing AI Assistant Implementation

The current Mochi assistant is implemented in
`modules/ai_assistant.py`.

### Main Functions

- `ask_ai(question)`: Sends a user question to the OpenAI API using the Mochi system instructions.
- `interpret_pantry_request(user_message)`: Converts a natural-language pantry request into a structured command.
- `handle_pantry_command(command, pantry)`: Executes supported pantry actions through the pantry manager.
- `confirm_pantry_command(command, pantry, confirmed)`: Requires confirmation before executing a pantry command.

### Current Capabilities

Mochi provides conversational assistance using an OpenAI model. The module also includes functions for interpreting and executing pantry addition requests.

### Observations

1. The assistant uses `gpt-4.1-mini`.
2. Pantry commands currently support the `add` action.
3. A confirmation function is available before executing pantry commands.
4. The general conversational function does not automatically retrieve live pantry inventory.
5. The quantity field is defined as a string in the structured response schema, but subsequent validation requires a numeric value.
6. Additional integration is needed to connect conversational responses with storage guidance, recipe recommendations, and recall checking.

### Phase 4 Development Focus

The next stage will test the existing functionality, investigate identified issues, and improve integration while preserving user confirmation and food safety requirements.


In [1]:
import sys
from pathlib import Path

current_directory = Path.cwd()

if (current_directory / "modules").is_dir():
    project_root = current_directory
elif (current_directory.parent / "modules").is_dir():
    project_root = current_directory.parent
elif (current_directory / "smartpantry-ai" / "modules").is_dir():
    project_root = current_directory / "smartpantry-ai"
else:
    raise FileNotFoundError("SmartPantry project folder not found.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("SmartPantry project path configured.")
print("Repository exists:", project_root.exists())
print("Project root:", project_root)

SmartPantry project path configured.
Repository exists: True
Project root: c:\Users\Samir\GitHub\smartpantry-ai


In [2]:
from modules import ai_assistant

print("Mochi module imported successfully!")

Mochi module imported successfully!


In [3]:
from pathlib import Path

module_path = Path(ai_assistant.__file__)
module_code = module_path.read_text(encoding="utf-8")

print("Module location:", module_path)
print(
    "Quantity schema corrected:",
    '"quantity": {"type": "number"}' in module_code
)

Module location: c:\Users\Samir\GitHub\smartpantry-ai\modules\ai_assistant.py
Quantity schema corrected: True


In [18]:
# Test 1: Reproduce the original quantity type mismatch

original_schema_type = "string"
original_quantity = "2"

# Reproduce the original numeric validation condition.
is_valid_quantity = (
    isinstance(original_quantity, (int, float))
    and not isinstance(original_quantity, bool)
)

print("Original schema type:", original_schema_type)
print("Simulated quantity:", repr(original_quantity))
print("Python quantity type:", type(original_quantity).__name__)
print("Accepted by original validation:", is_valid_quantity)

assert original_schema_type == "string"
assert is_valid_quantity is False

print("Test 1: PASS - Original mismatch reproduced")

Original schema type: string
Simulated quantity: '2'
Python quantity type: str
Accepted by original validation: False
Test 1: PASS - Original mismatch reproduced



## 2. AI Assistant Testing

### Test 1: Original Pantry Quantity Validation Mismatch

**Test Type:** Historical bug reproduction / Regression test

**Function Investigated:** `interpret_pantry_request()`

**Objective:** Reproduce the original mismatch between the JSON response schema and Python quantity validation.

**Original Test Input:** "Add 2 cartons of milk to my pantry"

**Original Implementation:**
- JSON schema quantity type: `string`
- Simulated quantity: `"2"`
- Python validation required an `int` or `float`

**Original Failure Observed:**

```text
Validation error: Quantity must be a positive, finite numhber.
```

**Updated Reproduction Test Output:**

```text
Original schema type: string
Simulated quantity: '2'
Python quantity type: str
Accepted by original validation: False
Test 1: PASS - Original mismatch reproduced
```

**Status:** PASS (historical mismatch successfully reproduced)

**Root Cause:** The original JSON schema requested a string, but Python validation required a numeric quantity.

**Correction:** Changed the quantity schema from `string` to `number` in `modules/ai_assistant.py`.

**API Usage:** None.

**Database Usage:** None.

**Conclusion:** The test independently reproduces the original data-type mismatch. The historical failure is preserved as evidence, while the current executable test passes.

**Limitation:** This reproduction tests the original validation condition independently rather than executing the original version of `interpret_pantry_request()`. Test 2 verifies the corrected module.


In [7]:
import importlib

importlib.reload(ai_assistant)

print("Mochi module reloaded successfully!")

Mochi module reloaded successfully!


In [5]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()
mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            result = ai_assistant.interpret_pantry_request(
                "Add 2 cartons of milk to my pantry"
            )
            print("Result:", result)

            # Verify that the corrected schema requests a number
            call_args = mock_openai.return_value.responses.create.call_args
            schema = call_args.kwargs["text"]["format"]["schema"]
            print("Schema quantity type:", schema["properties"]["quantity"]["type"])

        except ValueError as error:
            print("Validation error:", error)

Result: {'action': 'add', 'ingredient': 'milk', 'quantity': 2, 'unit': 'cartons'}
Schema quantity type: number


### Test 2: Pantry Quantity Validation Correction

**Test Type:** Mocked API response / Regression test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that the corrected JSON schema accepts a positive numeric quantity.

**Correction Applied:**
Changed the quantity property in `modules/ai_assistant.py` from `"type": "string"` to `"type": "number"`.

**Test Input:** "Add 2 cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: cartons

**Expected Result:** The function returns a valid structured pantry command, and the JSON schema specifies a numeric quantity.

**Actual Result:**
```text
Result: {'action': 'add', 'ingredient': 'milk', 'quantity': 2, 'unit': 'cartons'}
Schema quantity type: number

In [10]:
from unittest.mock import MagicMock

mock_pantry = MagicMock()

command = {
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}

result = ai_assistant.confirm_pantry_command(
    command,
    mock_pantry,
    confirmed=False
)

print("Result:", result)
print("Pantry add called:", mock_pantry.add_item.called)

assert result is None
assert mock_pantry.add_item.call_count == 0

print("Test 3: PASS")

Result: None
Pantry add called: False
Test 3: PASS


### Test 3: Pantry Addition Without User Confirmation

**Test Type:** Mocked unit test / Confirmation safeguard

**Function:** `confirm_pantry_command()`

**Objective:** Verify that Mochi does not modify pantry inventory when the user has not confirmed the requested action.

**Test Input:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: cartons
- User confirmation: False

**Expected Result:**
- The function returns `None`.
- The pantry manager's `add_item()` function is not called.
- No pantry modification occurs.

**Actual Result:**
```text
Result: None
Pantry add called: False
Test 3: PASS

In [11]:
from unittest.mock import MagicMock

mock_pantry = MagicMock()
mock_pantry.add_item.return_value = "Item added successfully"

command = {
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}

result = ai_assistant.confirm_pantry_command(
    command,
    mock_pantry,
    confirmed=True
)

print("Result:", result)
print("Pantry add called:", mock_pantry.add_item.called)

mock_pantry.add_item.assert_called_once_with(
    "milk", 2, "cartons"
)

print("Test 4: PASS")

Result: Item added successfully
Pantry add called: True
Test 4: PASS



### Test 4: Pantry Addition With User Confirmation

**Test Type:** Mocked unit test / Positive test

**Function:** `confirm_pantry_command()`

**Objective:** Verify that Mochi correctly passes an approved pantry addition command to the pantry manager.

**Test Input:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: cartons
- User confirmation: True

**Expected Result:**
- The pantry manager's `add_item()` method is called exactly once.
- The correct ingredient, quantity, and unit are passed.
- The function returns the pantry manager's result.

**Actual Result:**
```text
Result: Item added successfully
Pantry add called: True
Test 4: PASS
```

**Status:** PASS

**API Usage:** None.

**Database Usage:** None. A mocked pantry manager was used.

**Conclusion:** The confirmation function correctly delegates an approved pantry addition to the pantry manager.

**Limitation:** This test does not verify actual database persistence or the Streamlit confirmation interface.


In [12]:
from unittest.mock import MagicMock

mock_pantry = MagicMock()

invalid_command = {
    "action": "delete",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}

try:
    ai_assistant.handle_pantry_command(
        invalid_command,
        mock_pantry
    )

    print("Test 5: FAIL - Unsupported action was accepted")

except ValueError as error:
    print("Validation error:", error)

    mock_pantry.add_item.assert_not_called()

    print("Test 5: PASS")

Validation error: Unsupported pantry action.
Test 5: PASS



### Test 5: Unsupported Pantry Action

**Test Type:** Mocked unit test / Negative test

**Function:** `handle_pantry_command()`

**Objective:** Verify that Mochi rejects unsupported pantry actions without executing a pantry addition.

**Test Input:**
- Action: delete
- Ingredient: milk
- Quantity: 2
- Unit: cartons

**Expected Result:**
- The function raises a `ValueError`.
- The pantry manager's `add_item()` method is not called.

**Actual Result:**
```text
Validation error: Unsupported pantry action.
Test 5: PASS
```

**Status:** PASS

**API Usage:** None.

**Database Usage:** None. A mocked pantry manager was used.

**Conclusion:** The function correctly rejected the unsupported delete action.

**Limitation:** This test verifies that `add_item()` was not called. It does not establish whether future deletion functionality is implemented safely.


In [13]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "",
    "quantity": 2,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            result = ai_assistant.interpret_pantry_request(
                "Add 2 cartons to my pantry"
            )

            print("Test 6: FAIL - Missing ingredient accepted")

        except ValueError as error:
            print("Validation error:", error)
            print("Test 6: PASS")

Validation error: Ingredient name must be a non-empty string.
Test 6: PASS



### Test 6: Missing Ingredient Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects a pantry addition request when the ingredient name is missing.

**Test Input:** "Add 2 cartons to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: "" (empty string)
- Quantity: 2
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because the ingredient name is missing.

**Actual Result:**

```text
Validation error: Ingredient name must be a non-empty string.
Test 6: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected an empty ingredient name.

**Limitation:** The test confirms validation of an empty ingredient field, but does not verify whether a live AI model might invent an ingredient when interpreting an incomplete request.


In [14]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": 0,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            result = ai_assistant.interpret_pantry_request(
                "Add 0 cartons of milk to my pantry"
            )

            print("Test 7: FAIL - Zero quantity accepted")

        except ValueError as error:
            print("Validation error:", error)
            print("Test 7: PASS")

Validation error: Quantity must be a positive, finite numhber.
Test 7: PASS



### Test 7: Zero Quantity Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects pantry addition requests containing a quantity of zero.

**Test Input:** "Add 0 cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: 0
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because a pantry addition requires a positive quantity.

**Actual Result:**

```text
Validation error: Quantity must be a positive, finite numhber.
Test 7: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected a quantity of zero. This confirms that the positive-quantity validation works for this test case.

**Limitation:** Negative quantities, nonnumeric values, and non-finite numbers have not yet been tested.


In [15]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()
mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": -3,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            ai_assistant.interpret_pantry_request(
                "Add -3 cartons of milk to my pantry"
            )
        except ValueError as error:
            print("Validation error:", error)
            print("Test 8: PASS")
        else:
            print("Test 8: FAIL - Negative quantity accepted")

Validation error: Quantity must be a positive, finite numhber.
Test 8: PASS



### Test 8: Negative Quantity Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects pantry addition requests containing a negative quantity.

**Test Input:** "Add -3 cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: -3
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because pantry additions require positive quantities.

**Actual Result:**

```text
Validation error: Quantity must be a positive, finite numhber.
Test 8: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected a negative quantity, confirming that the positive-quantity validation handles both zero and negative values.

**Limitation:** Nonnumeric quantities, non-finite values, and missing measurement units require further testing.


In [16]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": "two",
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            ai_assistant.interpret_pantry_request(
                "Add two cartons of milk to my pantry"
            )

        except ValueError as error:
            print("Validation error:", error)
            print("Test 9: PASS")

        else:
            print("Test 9: FAIL - Nonnumeric quantity accepted")

Validation error: Quantity must be a positive, finite numhber.
Test 9: PASS



### Test 9: Nonnumeric Quantity Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects a quantity represented as nonnumeric text instead of a valid number.

**Test Input:** "Add two cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: "two"
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because the quantity is not a numeric value.

**Actual Result:**

```text
Validation error: Quantity must be a positive, finite numhber.
Test 9: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected a nonnumeric quantity, preventing an improperly formatted value from being returned as a valid pantry command.

**Limitation:** This test intentionally supplied a response that violates the corrected JSON schema. It does not evaluate how reliably the live OpenAI model converts written numbers into numeric values.


In [17]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": ""
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            ai_assistant.interpret_pantry_request(
                "Add 2 milk to my pantry"
            )

        except ValueError as error:
            print("Validation error:", error)
            print("Test 10: PASS")

        else:
            print("Test 10: FAIL - Missing unit accepted")

Validation error: Unit must be a non-empty string
Test 10: PASS



### Test 10: Missing Measurement Unit Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects a pantry addition request when the measurement unit is missing.

**Test Input:** "Add 2 milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: "" (empty string)

**Expected Result:** The function raises a `ValueError` because the measurement unit is missing.

**Actual Result:**

```text
Validation error: Unit must be a non-empty string
Test 10: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected an empty measurement unit.

**Limitation:** This test does not verify whether the live AI model asks the user to clarify a missing unit or whether it might invent one.




## 3. Testing Summary and Phase 4 Findings

### Test Results

Ten test scenarios were documented for Mochi's AI assistant.

The initial test identified a mismatch between the JSON schema and Python quantity validation. The schema was corrected to use a numeric type, and subsequent mocked tests passed.

The tests covered:
- Positive quantity validation
- User confirmation requirements
- Approved pantry additions
- Unsupported pantry actions
- Missing ingredients
- Zero and negative quantities
- Nonnumeric quantities
- Missing measurement units

All tests were rerun successfully in VS Code after the correction. No live OpenAI API calls or real database modifications were required.

### Current Limitations

These tests verify individual functions using mocked responses. They do not establish that:
- Mochi can retrieve a user's actual pantry inventory.
- Pantry additions persist correctly in the database.
- The Streamlit interface consistently requires confirmation.
- The live AI model reliably interprets incomplete or ambiguous requests.
- Mochi can retrieve verified storage, recipe, or recall information.

### Phase 4 Next Steps

Future development will focus on connecting Mochi to persistent pantry inventory, implementing safe confirmation workflows, integrating existing SmartPantry modules, and testing conversational functionality.

### Conclusion

The testing process identified and corrected a quantity-schema mismatch and verified the assistant's existing pantry command validation and confirmation safeguards.

The module now has a documented baseline for future integration testing.
